# HMS v7 — extra training seeds for the clean 50 s benchmark

**Inputs:** (1) HMS competition data; (2) `notebook86c832275b` **Version 5** (holds `results_v3/*_valpred.npy`, the seed-42 runs). To resume after the time budget, also attach this notebook's own output. GPU T4, **Save Version → Save & Run All**.

**What it does.** Retrains Mamba, GRU, Transformer and CNN at 50 s (two stages, identical recipe and folds) with training seeds 1042 and 2042, then compares the models on per-EEG KL averaged over 3 seeds (calibrated, patient-cluster bootstrap, Holm). This separates architecture effects from training noise in the clean 50 s benchmark.

**Time:** ≈ 10 min per fold for all four models together → ≈ 1.8 h for both seeds (+ ~3 min cache). Each run is saved as soon as it finishes.

In [1]:
import os, glob, json, math, time, random, shutil
import numpy as np
import pandas as pd

CFG = dict(
    DATA_ROOT=None,                       # None = auto-detect under /kaggle/input
    CACHE_DIR="/kaggle/working/cache",    # replaced in the v2 settings cell below
    RESULTS_DIR="/kaggle/working/results",
    FS=100,                               # target sampling rate (Hz)
    CTX_SEC=50,
    CACHE_CTX_SEC=100,
    JITTER_SEC=1.0,                       # train-time random shift of the window centre
    PATCH=25,                             # samples per token (0.25 s at 100 Hz)
    N_FOLDS=5, FOLDS=[0], SEED=42,
    MODELS=["mamba", "transformer", "gru", "cnn"],
    D_MODEL=128, N_LAYERS=4, D_STATE=64, HEADDIM=32, EXPAND=2, CHUNK=64, N_HEADS_ATTN=4,
    EPOCHS=15, BATCH=64, LR=1e-3, WD=0.05, WARMUP_FRAC=0.05, CLIP=1.0,
    TRAIN_MIN_VOTES=0,
    HIGH_VOTE=10,                         # threshold for the high-agreement evaluation subset
    NUM_WORKERS=4, AMP=True,
    USE_OFFICIAL_MAMBA2=False,
    RUN_BENCHMARK=True,
    BENCH_CTX=[50, 100, 200, 400, 600, 1200],
)

In [2]:
import pyarrow.parquet as pq
import multiprocessing as mp
from scipy.signal import butter, sosfiltfilt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, f1_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["SEED"])
MP = mp.get_context("fork")   # Kaggle is Linux; fork shares globals with workers
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "")

def find_root():
    if CFG["DATA_ROOT"]:
        return CFG["DATA_ROOT"]
    for pat in ["/kaggle/input/*/train.csv", "/kaggle/input/*/*/train.csv"]:
        for h in glob.glob(pat):
            if os.path.isdir(os.path.join(os.path.dirname(h), "train_eegs")):
                return os.path.dirname(h)
    raise FileNotFoundError("Attach the competition data: Add Input → Competitions → HMS - Harmful Brain Activity Classification")

ROOT = find_root()
print("data:", ROOT)

device: cuda Tesla T4
data: /kaggle/input/competitions/hms-harmful-brain-activity-classification


## Labels and patient-level folds

In [3]:
VOTES = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
CLASSES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]

df = pd.read_csv(f"{ROOT}/train.csv")
df["n_votes"] = df[VOTES].sum(axis=1)
P_COLS = [f"p_{c}" for c in CLASSES]
df[P_COLS] = df[VOTES].div(df["n_votes"], axis=0).to_numpy()
df["y"] = df[P_COLS].to_numpy().argmax(1)
df["center"] = df["eeg_label_offset_seconds"] + 25.0   # experts labelled the central 10 s of each 50 s window

sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
df["fold"] = -1
for k, (_, va) in enumerate(sgkf.split(df, df["y"], groups=df["patient_id"])):
    df.loc[df.index[va], "fold"] = k
assert df.groupby("patient_id")["fold"].nunique().max() == 1, "patient leaked across folds"

print(f"rows={len(df):,}  eegs={df.eeg_id.nunique():,}  patients={df.patient_id.nunique():,}")
print(df.groupby("fold").agg(rows=("eeg_id", "size"), eegs=("eeg_id", "nunique"), patients=("patient_id", "nunique")))
print(df.drop_duplicates("eeg_id")["expert_consensus"].value_counts())

rows=106,800  eegs=17,089  patients=1,950
       rows  eegs  patients
fold                       
0     17085  2795       383
1     25502  3689       386
2     21174  3585       394
3     22141  3238       397
4     20898  3782       390
expert_consensus
Other      7196
Seizure    2785
LPD        2577
GRDA       1828
GPD        1787
LRDA        916
Name: count, dtype: int64


## Preprocessing
18-channel bipolar double banana + midline (identical to I-CARE's 18 bipolar pairs). Band-pass 0.5–40 Hz at 200 Hz, then every second sample (100 Hz; the 40 Hz low-pass prevents aliasing). Amplitudes stay in µV (no per-window z-scoring), because absolute voltage — suppression, high-amplitude discharges — is clinically informative.

In [4]:
BIPOLAR = [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1"),    # left temporal chain
           ("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2"),    # right temporal chain
           ("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1"),    # left parasagittal chain
           ("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2"),    # right parasagittal chain
           ("Fz", "Cz"), ("Cz", "Pz")]                                 # midline
FLIP_PERM = [4, 5, 6, 7, 0, 1, 2, 3, 12, 13, 14, 15, 8, 9, 10, 11, 16, 17]   # left <-> right hemisphere
ELECTRODES = sorted({e for pair in BIPOLAR for e in pair})
N_CH = len(BIPOLAR)
SRC_FS = 200
SOS = butter(4, [0.5, 40.0], btype="band", fs=SRC_FS, output="sos")
CLIP_UV = 500.0

def load_bipolar(path):
    d = pd.read_parquet(path, columns=ELECTRODES)
    d = d.interpolate(limit_direction="both").fillna(0.0)
    a = d.to_numpy(np.float32)
    col = {c: i for i, c in enumerate(ELECTRODES)}
    x = np.stack([a[:, col[p]] - a[:, col[q]] for p, q in BIPOLAR], axis=1)
    x = sosfiltfilt(SOS, x, axis=0)
    return np.clip(x, -CLIP_UV, CLIP_UV).astype(np.float32)

def _nrows(eid):
    return pq.ParquetFile(f"{ROOT}/train_eegs/{eid}.parquet").metadata.num_rows

def plan_cache(df, half_sec):
    g = df.groupby("eeg_id")["center"].agg(["min", "max"]).reset_index()
    with MP.Pool(CFG["NUM_WORKERS"]) as pool:
        n = pool.map(_nrows, g["eeg_id"].tolist(), chunksize=64)
    g["n_src"] = n
    s0 = np.floor((g["min"] - half_sec) * SRC_FS).clip(lower=0).astype(np.int64)
    s0 -= s0 % 2
    s1 = np.minimum(g["n_src"], np.ceil((g["max"] + half_sec) * SRC_FS)).astype(np.int64)
    g["s0"], g["length"] = s0, ((s1 - s0 + 1) // 2).clip(lower=0)
    g["offset"] = np.concatenate([[0], np.cumsum(g["length"].to_numpy())[:-1]])
    return g[["eeg_id", "n_src", "s0", "length", "offset"]]

def _build_one(args):
    eid, s0, length = args
    x = load_bipolar(f"{ROOT}/train_eegs/{eid}.parquet")[s0::2][:length]
    out = np.zeros((length, N_CH), np.float16)
    out[:len(x)] = x
    return out

def build_cache(df):
    meta_path = f"{CFG['CACHE_DIR']}/meta.json"
    want = dict(cache_ctx=CFG["CACHE_CTX_SEC"], jitter=CFG["JITTER_SEC"], fs=CFG["FS"], n_eegs=int(df.eeg_id.nunique()))
    if os.path.exists(meta_path) and json.load(open(meta_path)) == want:
        print("cache found, reusing")
        return pd.read_csv(f"{CFG['CACHE_DIR']}/index.csv")
    half = CFG["CACHE_CTX_SEC"] / 2 + CFG["JITTER_SEC"]
    idx = plan_cache(df, half)
    total = int(idx["length"].sum())
    need = total * N_CH * 2
    free = shutil.disk_usage(CFG["CACHE_DIR"]).free
    print(f"cache: {total/CFG['FS']/3600:,.1f} h of EEG, {need/1e9:.2f} GB needed, {free/1e9:.1f} GB free")
    if need > 0.9 * free:
        raise RuntimeError("Not enough disk: lower CACHE_CTX_SEC or point CACHE_DIR at a larger disk")
    mm = np.lib.format.open_memmap(f"{CFG['CACHE_DIR']}/signals.npy", mode="w+", dtype=np.float16, shape=(total, N_CH))
    jobs = list(zip(idx["eeg_id"], idx["s0"], idx["length"]))
    t0 = time.time()
    with MP.Pool(CFG["NUM_WORKERS"]) as pool:
        for i, (x, off) in enumerate(zip(pool.imap(_build_one, jobs, chunksize=16), idx["offset"])):
            mm[off:off + len(x)] = x
            if i % 2000 == 0:
                print(f"  {i:,}/{len(jobs):,}  {time.time()-t0:.0f}s")
    mm.flush(); del mm
    idx.to_csv(f"{CFG['CACHE_DIR']}/index.csv", index=False)
    json.dump(want, open(meta_path, "w"))
    print(f"cache built in {time.time()-t0:.0f}s")
    return idx

## Dataset
Each item is a `CTX_SEC` window centred on the labelled 10 s. Positions outside the recording are zero-filled and flagged in a mask; the SSM treats masked steps as identity transitions (Δ = 0), the Transformer ignores them via key-padding, the readout never pools them.
Training samples one random sub-window per EEG per epoch, so long recordings with many overlapping sub-windows don't dominate. Validation uses the middle sub-window of each EEG (the competition test set also has one row per EEG).

In [5]:
class HMSDataset(Dataset):
    """Self-contained (no notebook globals in __getitem__), so it works with any worker start method."""
    def __init__(self, rows, ctx_sec, train):
        assert ctx_sec <= CFG["CACHE_CTX_SEC"], "CTX_SEC exceeds the cached context; rebuild the cache"
        self.eid = rows["eeg_id"].to_numpy()
        self.center = rows["center"].to_numpy(np.float64)
        self.p = rows[P_COLS].to_numpy(np.float32)
        self.y = rows["y"].to_numpy()
        self.fs, self.src_fs, self.n_ch = CFG["FS"], SRC_FS, N_CH
        self.n = int(ctx_sec * self.fs)
        assert self.n % CFG["PATCH"] == 0
        self.train, self.jitter, self.flip = train, CFG["JITTER_SEC"], np.array(FLIP_PERM)
        self.cache = {int(e): CACHE[int(e)] for e in np.unique(self.eid)}
        self.path = f"{CFG['CACHE_DIR']}/signals.npy"
        self.mm = None

    def __len__(self):
        return len(self.eid)

    def __getitem__(self, i):
        if self.mm is None:                      # open lazily so each worker gets its own handle
            self.mm = np.load(self.path, mmap_mode="r")
        off, s0, length = self.cache[int(self.eid[i])]
        c = self.center[i]
        if self.train:
            c += np.random.uniform(-self.jitter, self.jitter)
        i0 = int(round((c - self.n / self.fs / 2 - s0 / self.src_fs) * self.fs))
        x = np.zeros((self.n, self.n_ch), np.float32)
        m = np.zeros(self.n, np.float32)
        a, b = max(i0, 0), min(i0 + self.n, length)
        if b > a:
            x[a - i0:b - i0] = self.mm[off + a:off + b]
            m[a - i0:b - i0] = 1.0
        x = x.T / 50.0                           # ~unit scale while keeping absolute amplitude
        if self.train:
            if np.random.rand() < 0.5:
                x = x[self.flip]
            x = x * np.random.uniform(0.9, 1.1)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(m), torch.from_numpy(self.p[i]), int(self.y[i])

class OnePerEEG(Sampler):
    def __init__(self, eids, seed):
        order = np.argsort(eids, kind="stable")
        _, starts = np.unique(eids[order], return_index=True)
        self.groups = np.split(order, starts[1:])
        self.seed, self.epoch = seed, 0
    def __len__(self):
        return len(self.groups)
    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch); self.epoch += 1
        idx = np.array([g[rng.integers(len(g))] for g in self.groups])
        rng.shuffle(idx)
        return iter(idx.tolist())

def middle_row_per_eeg(rows):
    rows = rows.sort_values(["eeg_id", "eeg_label_offset_seconds"])
    g = rows.groupby("eeg_id")
    return rows[g.cumcount().to_numpy() == (g["eeg_id"].transform("size") // 2).to_numpy()]

## v7 settings

In [6]:
CFG.update(dict(
    CACHE_CTX_SEC=50, CACHE_DIR="/tmp/hms_cache", RESULTS_DIR="/kaggle/working/results_v7",
    PREV_RESULTS_GLOBS=["/kaggle/input/**/results_v7/*.json", "/kaggle/input/**/results_v7/*_valpred.npy"],   # resume
    V3_GLOBS=["/kaggle/input/**/results_v3/*_valpred.npy"],                                                  # seed-42 runs
    MODELS_V7=["mamba", "gru", "transformer", "cnn"], CTX=50, TRAIN_SEEDS=[1042, 2042],
    SWEEP_FOLDS=[0, 1, 2, 3, 4],
    EPOCHS=6, STAGE2_EPOCHS=3, STAGE2_LR=2e-4, STAGE2_MIN_VOTES=10, SELECT="last",
    DROPOUT=0.1, TOKENS_PER_BATCH=25600, GRAD_CKPT=False, USE_OFFICIAL_MAMBA2=False,
    TIME_BUDGET_H=6.0, B=5000, T_GRID=list(np.round(np.linspace(0.5, 3.0, 51), 3)),
))
CFG_OVERRIDES = {"MODELS_V7": ["mamba", "gru"], "TRAIN_SEEDS": [1042, 2042, 3042, 4042], "TIME_BUDGET_H": 2.5}
CFG.update(CFG_OVERRIDES)
T_START = time.time()
os.makedirs(CFG["CACHE_DIR"], exist_ok=True); os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)

In [7]:
if CFG["USE_OFFICIAL_MAMBA2"]:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
                    "causal-conv1d", "mamba-ssm"], check=False)

In [8]:
cache_idx = build_cache(df)
CACHE = {int(r.eeg_id): (int(r.offset), int(r.s0), int(r.length)) for r in cache_idx.itertuples()}

cache: 380.3 h of EEG, 4.93 GB needed, 1038.2 GB free
  0/17,089  1s
  2,000/17,089  22s
  4,000/17,089  45s
  6,000/17,089  67s
  8,000/17,089  89s
  10,000/17,089  111s
  12,000/17,089  132s
  14,000/17,089  153s
  16,000/17,089  174s
cache built in 187s


## Models
All models: `Stem` (conv + 0.25 s patching → tokens) → sequence mixer → readout (pool over the central 10 s ⊕ pool over the full context) → 6 logits. Only the mixer differs.

The SSM is a bidirectional Mamba-2 block. `ssd()` is the chunked state-space-duality algorithm (exact); `ssd_reference()` is the plain recurrence used to verify it. `Mamba2.step()` is the recurrent form used for streaming.

In [9]:
from torch.utils.checkpoint import checkpoint

def run_block(blk, *args):
    if CFG["GRAD_CKPT"] and blk.training and torch.is_grad_enabled():
        return checkpoint(blk, *args, use_reentrant=False)
    return blk(*args)

def segsum(x):
    """x: (..., T) log-decays -> (..., T, T) with out[i, j] = sum_{k=j+1..i} x_k for i >= j, else -inf."""
    T = x.size(-1)
    x = x[..., None].expand(*x.shape, T)
    low = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device), -1)
    x = x.masked_fill(~low, 0)
    s = torch.cumsum(x, dim=-2)
    keep = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device), 0)
    return s.masked_fill(~keep, -torch.inf)

def ssd(X, A, B, C, chunk):
    """Selective SSM, chunked SSD form.  h_t = exp(A_t) h_{t-1} + X_t B_t^T,  y_t = h_t C_t
    X: (b, T, h, p) input already multiplied by dt;  A: (b, T, h) = dt * A (<= 0);
    B, C: (b, T, n), shared by all heads (Mamba-2 with ngroups = 1)."""
    b, T, h, p = X.shape
    pad = (-T) % chunk
    if pad:
        X = F.pad(X, (0, 0, 0, 0, 0, pad))
        B, C = F.pad(B, (0, 0, 0, pad)), F.pad(C, (0, 0, 0, pad))
        A = F.pad(A, (0, 0, 0, pad))
    c = (T + pad) // chunk
    X = X.reshape(b, c, chunk, h, p)
    B = B.reshape(b, c, chunk, -1)
    C = C.reshape(b, c, chunk, -1)
    A = A.reshape(b, c, chunk, h).permute(0, 3, 1, 2)            # (b, h, c, l)
    A_cs = torch.cumsum(A, dim=-1)
    L = torch.exp(segsum(A))                                     # (b, h, c, l, s)
    CB = torch.einsum("bcln,bcsn->bcls", C, B)
    Y_diag = torch.einsum("bcls,bhcls,bcshp->bclhp", CB, L, X)
    decay_states = torch.exp(A_cs[..., -1:] - A_cs)
    states = torch.einsum("bcln,bhcl,bclhp->bchpn", B, decay_states, X)
    states = torch.cat([torch.zeros_like(states[:, :1]), states], dim=1)
    decay_chunk = torch.exp(segsum(F.pad(A_cs[..., -1], (1, 0))))
    states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)[:, :-1]
    Y_off = torch.einsum("bcln,bchpn,bhcl->bclhp", C, states, torch.exp(A_cs))
    return (Y_diag + Y_off).reshape(b, c * chunk, h, p)[:, :T]

def ssd_reference(X, A, B, C):
    b, T, h, p = X.shape
    state = X.new_zeros(b, h, p, B.size(-1))
    ys = []
    for t in range(T):
        state = torch.exp(A[:, t])[..., None, None] * state + X[:, t, :, :, None] * B[:, t, None, None, :]
        ys.append(torch.einsum("bhpn,bn->bhp", state, C[:, t]))
    return torch.stack(ys, 1)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.w

class Mamba2(nn.Module):
    """Mamba-2 mixer (pure PyTorch). Masked steps get dt = 0: no decay, no input -> state passes through unchanged."""
    def __init__(self, d, d_state, headdim, expand, chunk, d_conv=4):
        super().__init__()
        self.di, self.n, self.p, self.chunk = expand * d, d_state, headdim, chunk
        self.h = self.di // headdim
        self.in_proj = nn.Linear(d, 2 * self.di + 2 * d_state + self.h, bias=False)
        cd = self.di + 2 * d_state
        self.conv = nn.Conv1d(cd, cd, d_conv, groups=cd, padding=d_conv - 1)
        dt = torch.exp(torch.rand(self.h) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))     # inverse softplus
        self.A_log = nn.Parameter(torch.log(torch.empty(self.h).uniform_(1, 16)))
        self.D = nn.Parameter(torch.ones(self.h))
        self.norm = RMSNorm(self.di)
        self.out_proj = nn.Linear(self.di, d, bias=False)

    def forward(self, u, m):
        b, T, _ = u.shape
        z, xBC, dt = torch.split(self.in_proj(u), [self.di, self.di + 2 * self.n, self.h], dim=-1)
        xBC = xBC * m[..., None]
        xBC = F.silu(self.conv(xBC.transpose(1, 2))[..., :T].transpose(1, 2))
        x, Bm, Cm = torch.split(xBC, [self.di, self.n, self.n], dim=-1)
        with torch.autocast(device_type=u.device.type, enabled=False):
            dt = F.softplus(dt.float() + self.dt_bias) * m[..., None]
            A = -torch.exp(self.A_log.float())
            x = x.float().reshape(b, T, self.h, self.p)
            y = ssd(x * dt[..., None], A * dt, Bm.float(), Cm.float(), self.chunk) + x * self.D[:, None]
        y = y.reshape(b, T, self.di).to(u.dtype)
        return self.out_proj(self.norm(y * F.silu(z)))

    def init_state(self, b, device):
        cd = self.di + 2 * self.n
        return (torch.zeros(b, cd, self.conv.kernel_size[0], device=device),
                torch.zeros(b, self.h, self.p, self.n, device=device))

    def step(self, u, state):
        """One token, u: (b, d). Fixed-size state: conv window (b, cd, d_conv) + SSM state (b, h, p, n)."""
        conv_st, ssm_st = state
        b = u.size(0)
        z, xBC, dt = torch.split(self.in_proj(u), [self.di, self.di + 2 * self.n, self.h], dim=-1)
        conv_st = torch.roll(conv_st, -1, dims=-1)
        conv_st[..., -1] = xBC.float()
        xBC = F.silu((conv_st * self.conv.weight[:, 0, :]).sum(-1) + self.conv.bias)
        x, Bm, Cm = torch.split(xBC, [self.di, self.n, self.n], dim=-1)
        dt = F.softplus(dt.float() + self.dt_bias)
        dA = torch.exp(dt * -torch.exp(self.A_log.float()))
        x = x.float().reshape(b, self.h, self.p)
        ssm_st = ssm_st * dA[..., None, None] + (x * dt[..., None])[..., None] * Bm.float()[:, None, None, :]
        y = torch.einsum("bhpn,bn->bhp", ssm_st, Cm.float()) + x * self.D[:, None]
        y = y.reshape(b, self.di).to(u.dtype)
        return self.out_proj(self.norm(y * F.silu(z))), (conv_st, ssm_st)

class OfficialMamba2(nn.Module):
    def __init__(self, d, d_state, headdim, expand, chunk):
        super().__init__()
        from mamba_ssm import Mamba2 as M2
        self.m = M2(d_model=d, d_state=d_state, headdim=headdim, expand=expand, chunk_size=chunk)
    def forward(self, u, m):
        return self.m(u * m[..., None])

class BiSSMBlock(nn.Module):
    def __init__(self, d, bidir=True):
        super().__init__()
        cls = OfficialMamba2 if CFG["USE_OFFICIAL_MAMBA2"] else Mamba2
        mk = lambda: cls(d, CFG["D_STATE"], CFG["HEADDIM"], CFG["EXPAND"], CFG["CHUNK"])
        self.norm, self.fwd = nn.LayerNorm(d), mk()
        self.bwd = mk() if bidir else None
        self.drop = nn.Dropout(CFG["DROPOUT"])
    def forward(self, h, m):
        x = self.norm(h)
        mf = m.float()
        y = self.fwd(x, mf)
        if self.bwd is not None:
            y = y + self.bwd(x.flip(1), mf.flip(1)).flip(1)
        return h + self.drop(y)

class SSMMixer(nn.Module):
    """bidir=True: L bidirectional blocks. bidir=False: 2L forward-only blocks (same parameter count), causal over the sequence."""
    def __init__(self, d, L, bidir=True):
        super().__init__()
        self.blocks = nn.ModuleList([BiSSMBlock(d, bidir) for _ in range(L if bidir else 2 * L)])
    def forward(self, h, m):
        for blk in self.blocks:
            h = run_block(blk, h, m)
        return h

def sinusoid(n, d, device):
    pos = torch.arange(n, device=device, dtype=torch.float32)[:, None]
    div = torch.exp(torch.arange(0, d, 2, device=device, dtype=torch.float32) * (-math.log(10000.0) / d))
    pe = torch.zeros(n, d, device=device)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
    return pe

class TransformerMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__()
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(d, CFG["N_HEADS_ATTN"], 4 * d, dropout=0.1, activation="gelu",
                                                                batch_first=True, norm_first=True) for _ in range(L)])
    def forward(self, h, m):
        h = h + sinusoid(h.size(1), h.size(2), h.device).to(h.dtype)
        kpm = ~m
        for layer in self.layers:
            h = run_block(layer, h, None, kpm)
        return h

class GRUMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__()
        self.rnn = nn.GRU(d, d, L, batch_first=True, bidirectional=True, dropout=0.1)
        self.proj = nn.Linear(2 * d, d)
    def forward(self, h, m):
        return self.proj(run_block(self.rnn, h * m[..., None])[0])

class ConvBlock(nn.Module):
    def __init__(self, d, dil):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.dw = nn.Conv1d(d, d, 9, padding=4 * dil, dilation=dil, groups=d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(CFG["DROPOUT"])
    def forward(self, h, m):
        x = (self.norm(h) * m[..., None]).transpose(1, 2)
        return h + self.drop(self.mlp(self.dw(x).transpose(1, 2)))

class CNNMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__(); self.blocks = nn.ModuleList([ConvBlock(d, 2 ** (i % 6)) for i in range(2 * L)])
    def forward(self, h, m):
        for blk in self.blocks:
            h = run_block(blk, h, m)
        return h

MIXERS = dict(mamba=SSMMixer, mamba_causal=lambda d, L: SSMMixer(d, L, bidir=False),
              transformer=TransformerMixer, gru=GRUMixer, cnn=CNNMixer)

class Stem(nn.Module):
    def __init__(self, d, patch):
        super().__init__()
        self.patch = patch
        self.c1 = nn.Conv1d(N_CH, d, 7, padding=3)
        self.c2 = nn.Conv1d(d, d, patch, stride=patch)
        self.norm = nn.LayerNorm(d)
    def forward(self, x, m):
        h = self.c2(F.gelu(self.c1(x * m[:, None]))).transpose(1, 2)
        tm = F.avg_pool1d(m[:, None], self.patch, self.patch)[:, 0] > 0.5
        return self.norm(h) * tm[..., None], tm

class Net(nn.Module):
    def __init__(self, mixer):
        super().__init__()
        d = CFG["D_MODEL"]
        self.stem = Stem(d, CFG["PATCH"])
        self.mixer = MIXERS[mixer](d, CFG["N_LAYERS"])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, len(CLASSES)))
        self.half_center = int(5 * CFG["FS"] / CFG["PATCH"])     # tokens in 5 s

    def forward(self, x, m):
        h, tm = self.stem(x, m)
        h = self.norm(self.mixer(h, tm))
        c = h.size(1) // 2
        cm = torch.zeros_like(tm)
        cm[:, max(c - self.half_center, 0):c + self.half_center] = True
        cm &= tm
        def pool(mask):
            w = mask.to(h.dtype)[..., None]
            return (h * w).sum(1) / w.sum(1).clamp(min=1)
        return self.head(torch.cat([pool(cm), pool(tm)], dim=-1))

def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Sanity checks
1. Chunked SSD equals the step-by-step recurrence.
2. Edge padding is exact: zero-filled context outside the recording (dt = 0) gives the same outputs as the unpadded sequence.
3. The recurrent `step()` (streaming form) reproduces the parallel forward pass token by token.
4. Gradient checkpointing does not change outputs.
5. The causal SSM mixer really is causal.
6. If official kernels are on: they agree with the pure implementation on the same weights' task (shape check + finite output).

In [10]:
with torch.no_grad():
    g = torch.Generator().manual_seed(0)
    b, T, h, p, n = 2, 150, 3, 4, 5
    X = torch.randn(b, T, h, p, generator=g, dtype=torch.float64)
    A = -torch.rand(b, T, h, generator=g, dtype=torch.float64)
    B = torch.randn(b, T, n, generator=g, dtype=torch.float64)
    C = torch.randn(b, T, n, generator=g, dtype=torch.float64)
    err = (ssd(X, A, B, C, chunk=32) - ssd_reference(X, A, B, C)).abs().max().item()
    print(f"1. SSD vs recurrence max abs err: {err:.2e}")
    assert err < 1e-8

    blk = Mamba2(16, 8, 4, 2, 16).eval()
    u = torch.randn(1, 40, 16, generator=g)
    lead, trail = 12, 9
    u_pad = torch.cat([torch.randn(1, lead, 16, generator=g), u, torch.randn(1, trail, 16, generator=g)], 1)
    m_pad = torch.cat([torch.zeros(1, lead), torch.ones(1, 40), torch.zeros(1, trail)], 1)
    edge_err = (blk(u, torch.ones(1, 40)) - blk(u_pad, m_pad)[:, lead:lead + 40]).abs().max().item()
    print(f"2. edge-padding invariance err: {edge_err:.2e}")
    assert edge_err < 1e-5
    # A masked gap in the *middle* of a sequence is skipped by the SSM (dt = 0) but still touches the
    # 4-tap causal conv for the next 3 tokens. Irrelevant for HMS (gaps only at edges); handle it for I-CARE.

    st, ys = blk.init_state(1, "cpu"), []
    for t in range(40):
        y_t, st = blk.step(u[:, t], st)
        ys.append(y_t)
    step_err = (blk(u, torch.ones(1, 40)) - torch.stack(ys, 1)).abs().max().item()
    print(f"3. recurrent step vs parallel forward err: {step_err:.2e}")
    assert step_err < 1e-5

with torch.no_grad():
    cm_ = SSMMixer(32, 1, bidir=False).eval()
    h1 = torch.randn(1, 60, 32, generator=g); h2 = h1.clone(); h2[:, 40:] = torch.randn(1, 20, 32, generator=g)
    mk_ = torch.ones(1, 60, dtype=torch.bool)
    caus_err = (cm_(h1, mk_)[:, :40] - cm_(h2, mk_)[:, :40]).abs().max().item()
    print(f"5. causal mixer: change in outputs before t=40 when inputs after t=40 change: {caus_err:.2e}")
    assert caus_err < 1e-6

_ckpt = CFG["GRAD_CKPT"]
for name in ["mamba", "transformer"]:
    torch.manual_seed(0)
    net = Net(name).train()
    for mod in net.modules():
        if isinstance(mod, nn.Dropout): mod.p = 0.0
        if isinstance(mod, nn.MultiheadAttention): mod.dropout = 0.0   # attention dropout is a float, not a module
    xx, mm_ = torch.randn(2, N_CH, 1000), torch.ones(2, 1000)
    CFG["GRAD_CKPT"] = False; o1 = net(xx, mm_)
    CFG["GRAD_CKPT"] = True;  o2 = net(xx, mm_)
    print(f"4. grad-checkpoint output diff ({name}): {(o1 - o2).abs().max().item():.2e}")
    assert torch.allclose(o1, o2, atol=1e-5)
CFG["GRAD_CKPT"] = _ckpt

if CFG["USE_OFFICIAL_MAMBA2"]:
    assert DEV.type == "cuda", "official kernels need a GPU"
    net = Net("mamba").to(DEV).eval()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        out = net(torch.randn(2, N_CH, 5000, device=DEV), torch.ones(2, 5000, device=DEV))
    assert out.shape == (2, len(CLASSES)) and torch.isfinite(out).all()
    print("6. official mamba-ssm kernels: OK")

for name in MIXERS:
    print(f"{name:12s} params: {n_params(Net(name)):,}")

1. SSD vs recurrence max abs err: 1.95e-14
2. edge-padding invariance err: 1.19e-07
3. recurrent step vs parallel forward err: 3.58e-07
5. causal mixer: change in outputs before t=40 when inputs after t=40 change: 0.00e+00
4. grad-checkpoint output diff (mamba): 0.00e+00
4. grad-checkpoint output diff (transformer): 0.00e+00
mamba        params: 1,404,486
mamba_causal params: 1,405,510
transformer  params: 1,253,254
gru          params: 1,580,550
cnn          params: 1,526,150


## Training: two stages, token-budget micro-batching, resumable sweep
Stage 1 trains on all rows (best epoch by val KL on all EEGs). Stage 2 starts from the stage-1 best weights and fine-tunes on rows with ≥ `STAGE2_MIN_VOTES` votes at a lower LR (best epoch by val KL on the ≥10-vote subset). Both are saved. With `SELECT="last"` the reported metrics are those of the final epoch of each stage (no epoch picking on the test fold); the per-epoch history is kept in each run's JSON.

In [11]:
HV = f"votes>={CFG['HIGH_VOTE']}"

def kl_div(P, T, eps=1e-15):
    return float((T * (np.log(T + eps) - np.log(P + eps))).sum(1).mean())

def metrics(P, T, y, nv):
    out = {}
    for tag, sel in [("all", np.ones(len(y), bool)), (HV, nv >= CFG["HIGH_VOTE"])]:
        if sel.sum() == 0:
            continue
        p, t, yy = P[sel], T[sel], y[sel]
        r = dict(n=int(sel.sum()), kl=kl_div(p, t), acc=float((p.argmax(1) == yy).mean()),
                 f1_macro=float(f1_score(yy, p.argmax(1), average="macro", labels=range(len(CLASSES)), zero_division=0)))
        try:
            r["auroc_macro"] = float(roc_auc_score(yy, p, multi_class="ovr", average="macro", labels=list(range(len(CLASSES)))))
        except ValueError:
            r["auroc_macro"] = float("nan")
        out[tag] = r
    return out

@torch.no_grad()
def predict(model, loader):
    model.eval(); P = []
    for x, m, _, _ in loader:
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=CFG["AMP"] and DEV.type == "cuda"):
            P.append(torch.softmax(model(x.to(DEV), m.to(DEV)).float(), -1).cpu())
    return torch.cat(P).numpy()

def param_groups(model):
    decay, no_decay = [], []
    for n_, p in model.named_parameters():
        (no_decay if p.ndim < 2 or n_.endswith(("A_log", "D", "dt_bias")) else decay).append(p)
    return [dict(params=decay, weight_decay=CFG["WD"]), dict(params=no_decay, weight_decay=0.0)]

def make_loader(ds, batch, sampler=None, drop_last=False):
    kw = dict(num_workers=CFG["NUM_WORKERS"], pin_memory=DEV.type == "cuda", persistent_workers=CFG["NUM_WORKERS"] > 0,
              multiprocessing_context=MP if CFG["NUM_WORKERS"] > 0 else None)
    return DataLoader(ds, batch, sampler=sampler, shuffle=False, drop_last=drop_last, **kw)

def fit(model, dl_tr, dl_va, ds_va, va_votes, accum, epochs, lr, warm_frac, select, tag):
    opt = torch.optim.AdamW(param_groups(model), lr=lr)
    total = max(1, epochs * (len(dl_tr) // accum))
    warm = int(warm_frac * total)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (min(1.0, (s + 1) / warm) if warm else 1.0) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    use_amp = CFG["AMP"] and DEV.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best, best_state, hist = None, None, []
    for ep in range(epochs):
        model.train(); t0 = time.time(); tl, nb = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for i, (x, m, p, _) in enumerate(dl_tr):
            x, m, p = x.to(DEV, non_blocking=True), m.to(DEV, non_blocking=True), p.to(DEV, non_blocking=True)
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x, m)
            loss = F.kl_div(F.log_softmax(logits.float(), -1), p, reduction="batchmean")
            scaler.scale(loss / accum).backward()
            tl += loss.item(); nb += 1
            if (i + 1) % accum == 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), CFG["CLIP"])
                scaler.step(opt); scaler.update(); sched.step()
                opt.zero_grad(set_to_none=True)
        P = predict(model, dl_va)
        res = metrics(P, ds_va.p, ds_va.y, va_votes)
        hist.append(dict(epoch=ep, train_kl=tl / max(nb, 1), **{f"{k}_{m_}": v for k, d in res.items() for m_, v in d.items()}))
        a, hv = res["all"], res.get(HV, {"kl": float("nan")})
        print(f"  {tag} ep {ep:2d} | train KL {tl/max(nb,1):.4f} | val KL all {a['kl']:.4f}  {HV} {hv['kl']:.4f} | "
              f"acc {a['acc']:.3f} F1 {a['f1_macro']:.3f} AUROC {a['auroc_macro']:.3f} | {time.time()-t0:.0f}s")
        key = res[select]["kl"] if select in res else a["kl"]
        take = (ep == epochs - 1) if CFG["SELECT"] == "last" else (best is None or key < best["select_kl"])
        if take:
            best = dict(epoch=ep, select=select, select_kl=key, metrics=res)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            np.save(f"{CFG['RESULTS_DIR']}/{tag}_valpred.npy", P)
    return best, best_state, hist

def train_run(model_name, ctx, fold, seed):
    tag = f"{model_name}_ctx{ctx}_f{fold}_seed{seed}"
    seed_all(seed + fold)
    tok = ctx * CFG["FS"] // CFG["PATCH"]
    micro = max(1, min(CFG["BATCH"], CFG["TOKENS_PER_BATCH"] // tok))
    accum = math.ceil(CFG["BATCH"] / micro)
    tr = df[df.fold != fold].reset_index(drop=True)
    tr2 = tr[tr.n_votes >= CFG["STAGE2_MIN_VOTES"]].reset_index(drop=True)
    va = middle_row_per_eeg(df[df.fold == fold]).reset_index(drop=True)
    ds_va = HMSDataset(va, ctx, False)
    dl_va = make_loader(ds_va, micro * 2)
    va_votes = va["n_votes"].to_numpy()

    model = Net(model_name).to(DEV)
    print(f"\n=== {tag} | params {n_params(model):,} | micro-batch {micro} x accum {accum} | "
          f"train eegs {tr.eeg_id.nunique():,} (stage 2: {tr2.eeg_id.nunique():,}) | val eegs {len(va):,}")
    t_run = time.time()

    ds1 = HMSDataset(tr, ctx, True)
    dl1 = make_loader(ds1, micro, OnePerEEG(tr["eeg_id"].to_numpy(), seed + fold), drop_last=True)
    best1, state1, hist1 = fit(model, dl1, dl_va, ds_va, va_votes, accum, CFG["EPOCHS"], CFG["LR"], CFG["WARMUP_FRAC"], "all", f"{tag}_s1")
    torch.save(state1, f"{CFG['RESULTS_DIR']}/{tag}_s1.pt")

    best2, hist2 = None, []
    if CFG["STAGE2_EPOCHS"] > 0 and len(tr2):
        model.load_state_dict(state1)
        ds2 = HMSDataset(tr2, ctx, True)
        dl2 = make_loader(ds2, micro, OnePerEEG(tr2["eeg_id"].to_numpy(), seed + 100 + fold), drop_last=True)
        best2, state2, hist2 = fit(model, dl2, dl_va, ds_va, va_votes, accum, CFG["STAGE2_EPOCHS"], CFG["STAGE2_LR"], 0.0, HV, f"{tag}_s2")
        torch.save(state2, f"{CFG['RESULTS_DIR']}/{tag}_s2.pt")

    out = dict(model=model_name, fold=fold, seed=seed, ctx_sec=ctx, params=n_params(model), micro_batch=micro, accum=accum,
               minutes=(time.time() - t_run) / 60, stage1=best1, stage2=best2, history1=hist1, history2=hist2,
               cfg={k: v for k, v in CFG.items() if not k.startswith(("BENCH", "STREAM", "PREV"))})
    json.dump(out, open(f"{CFG['RESULTS_DIR']}/{tag}.json", "w"), indent=1)
    print(f"=== {tag} done in {out['minutes']:.1f} min")
    del model
    if DEV.type == "cuda": torch.cuda.empty_cache()
    return out

n_copied = 0
for pat in CFG["PREV_RESULTS_GLOBS"]:
    for f_ in glob.glob(pat, recursive=True):
        dst = os.path.join(CFG["RESULTS_DIR"], os.path.basename(f_))
        if not os.path.exists(dst):
            shutil.copy(f_, dst); n_copied += 1
print(f"resumed {n_copied} files from earlier v7 outputs")
V3 = {}
for pat in CFG["V3_GLOBS"]:
    for fp in glob.glob(pat, recursive=True):
        V3.setdefault(os.path.basename(fp), fp)
print(f"found {len(V3)} seed-42 prediction files")

RUNS = [(m_, f_, s_) for s_ in CFG["TRAIN_SEEDS"] for f_ in CFG["SWEEP_FOLDS"] for m_ in CFG["MODELS_V7"]]
print(f"{len(RUNS)} runs planned")
for m_, f_, s_ in RUNS:
    tag = f"{m_}_ctx{CFG['CTX']}_f{f_}_seed{s_}"
    if os.path.exists(f"{CFG['RESULTS_DIR']}/{tag}.json"):
        print(f"skip {tag} (already done)"); continue
    if (time.time() - T_START) / 3600 > CFG["TIME_BUDGET_H"]:
        print(f"time budget reached; stopping before {tag}. Attach this output and rerun to continue."); break
    train_run(m_, CFG["CTX"], f_, s_)

resumed 120 files from earlier v7 outputs
found 120 seed-42 prediction files
40 runs planned
skip mamba_ctx50_f0_seed1042 (already done)
skip gru_ctx50_f0_seed1042 (already done)
skip mamba_ctx50_f1_seed1042 (already done)
skip gru_ctx50_f1_seed1042 (already done)
skip mamba_ctx50_f2_seed1042 (already done)
skip gru_ctx50_f2_seed1042 (already done)
skip mamba_ctx50_f3_seed1042 (already done)
skip gru_ctx50_f3_seed1042 (already done)
skip mamba_ctx50_f4_seed1042 (already done)
skip gru_ctx50_f4_seed1042 (already done)
skip mamba_ctx50_f0_seed2042 (already done)
skip gru_ctx50_f0_seed2042 (already done)
skip mamba_ctx50_f1_seed2042 (already done)
skip gru_ctx50_f1_seed2042 (already done)
skip mamba_ctx50_f2_seed2042 (already done)
skip gru_ctx50_f2_seed2042 (already done)
skip mamba_ctx50_f3_seed2042 (already done)
skip gru_ctx50_f3_seed2042 (already done)
skip mamba_ctx50_f4_seed2042 (already done)
skip gru_ctx50_f4_seed2042 (already done)

=== mamba_ctx50_f0_seed3042 | params 1,404,486

## Results across training seeds (pooled out-of-fold, calibrated)
Seed 42 = the original runs. Stage-1 models are scored on all EEGs and on ≥10-vote EEGs, stage-2 models on ≥10-vote EEGs. Temperatures are cross-fitted (fitted on the other 4 folds, on the subset the stage is evaluated on). Main test: per-EEG KL averaged over the seeds both models share; patient-cluster bootstrap, Holm across the main tests.

In [12]:
VA_ROWS = {f_: middle_row_per_eeg(df[df.fold == f_]).reset_index(drop=True) for f_ in range(CFG["N_FOLDS"])}
OOF_ROWS = pd.concat([VA_ROWS[f_] for f_ in range(CFG["N_FOLDS"])], ignore_index=True)
T_OOF = OOF_ROWS[P_COLS].to_numpy(np.float64)
NV_OOF = OOF_ROWS["n_votes"].to_numpy()
PID_OOF = OOF_ROWS["patient_id"].to_numpy()
FOLD_OOF = np.concatenate([np.full(len(VA_ROWS[f_]), f_) for f_ in range(CFG["N_FOLDS"])])
HVM = NV_OOF >= CFG["HIGH_VOTE"]

def load_oof(model, stage, seed):
    parts = []
    for f_ in range(CFG["N_FOLDS"]):
        if seed == 42:
            b = f"{model}_ctx{CFG['CTX']}_f{f_}_{stage}_valpred.npy"; fp = V3.get(b)
        else:
            fp = f"{CFG['RESULTS_DIR']}/{model}_ctx{CFG['CTX']}_f{f_}_seed{seed}_{stage}_valpred.npy"
            fp = fp if os.path.exists(fp) else None
        if fp is None:
            return None
        P = np.load(fp); assert len(P) == len(VA_ROWS[f_]), fp
        parts.append(P)
    return np.concatenate(parts).astype(np.float64)

def per_eeg_kl(P, eps=1e-15):
    return (T_OOF * (np.log(T_OOF + eps) - np.log(P + eps))).sum(1)

def temper(P, t):
    L_ = np.log(np.clip(P, 1e-12, 1)) / t; L_ -= L_.max(1, keepdims=True); E = np.exp(L_)
    return E / E.sum(1, keepdims=True)

def calibrate(P, fit_mask):
    out = np.empty_like(P)
    for f_ in range(CFG["N_FOLDS"]):
        tr_, te_ = (FOLD_OOF != f_) & fit_mask, FOLD_OOF == f_
        t = min(CFG["T_GRID"], key=lambda t: per_eeg_kl_sub(temper(P[tr_], t), tr_).mean())
        out[te_] = temper(P[te_], t)
    return out

def per_eeg_kl_sub(P, mask, eps=1e-15):
    T_ = T_OOF[mask]
    return (T_ * (np.log(T_ + eps) - np.log(P + eps))).sum(1)

def cluster_bootstrap_diff(d, pid, B=CFG["B"], seed=0):
    up, inv = np.unique(pid, return_inverse=True)
    s_ = np.bincount(inv, weights=d, minlength=len(up)); n_ = np.bincount(inv, minlength=len(up)).astype(np.float64)
    W = np.random.default_rng(seed).multinomial(len(up), np.full(len(up), 1.0 / len(up)), size=B).astype(np.float64)
    boot = (W @ s_) / (W @ n_)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return d.mean(), lo, hi, max(min(1.0, 2 * min((boot <= 0).mean(), (boot >= 0).mean())), 1.0 / B)

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); out = np.empty_like(p); run = 0.0
    for i_, ix in enumerate(o):
        run = max(run, min(1.0, (len(p) - i_) * p[ix])); out[ix] = run
    return out

SEEDS_ = [42] + list(CFG["TRAIN_SEEDS"])
CELLS = [("s1", "all EEGs", np.ones(len(OOF_ROWS), bool), np.ones(len(OOF_ROWS), bool)),
         ("s1", ">=10 votes", HVM, np.ones(len(OOF_ROWS), bool)),
         ("s2", ">=10 votes", HVM, HVM)]                       # (stage, subset name, eval mask, calibration-fit mask)
KL = {}
for m_ in CFG["MODELS_V7"]:
    for st in ("s1", "s2"):
        for s_ in SEEDS_:
            P = load_oof(m_, st, s_)
            if P is None:
                continue
            for st_c, sname, ev, fit in CELLS:
                if st_c == st:
                    KL[(m_, st, s_, sname)] = per_eeg_kl(calibrate(P, fit))
print(f"{len(KL)} calibrated (model, stage, seed, subset) sets; seeds found per model/stage:")
for m_ in CFG["MODELS_V7"]:
    for st in ("s1", "s2"):
        print(f"  {m_} {st}: {[s_ for s_ in SEEDS_ if (m_, st, s_, CELLS[0][1] if st == 's1' else '>=10 votes') in KL]}")

rows = []
for st, sname, ev, _ in CELLS:
    for m_ in CFG["MODELS_V7"]:
        for s_ in SEEDS_:
            k = KL.get((m_, st, s_, sname))
            if k is not None:
                rows.append(dict(stage=st, subset=sname, model=m_, seed=s_, kl=k[ev].mean()))
per_seed = pd.DataFrame(rows)
per_seed.to_csv(f"{CFG['RESULTS_DIR']}/v7_per_seed.csv", index=False)
if len(per_seed):
    agg = per_seed.groupby(["stage", "subset", "model"], sort=False).agg(n_seeds=("seed", "size"), kl_mean=("kl", "mean"), kl_sd=("kl", "std"),
                                                                        kl_min=("kl", "min"), kl_max=("kl", "max")).reset_index()
    agg.to_csv(f"{CFG['RESULTS_DIR']}/v7_seed_summary.csv", index=False)
    print("### Calibrated pooled OOF KL per seed\n" + per_seed.pivot_table(index=["stage", "subset", "model"], columns="seed", values="kl", sort=False).round(4).to_string())
    print("\n### Mean and SD across seeds\n" + agg.round(4).to_string(index=False))

30 calibrated (model, stage, seed, subset) sets; seeds found per model/stage:
  mamba s1: [42, 1042, 2042, 3042, 4042]
  mamba s2: [42, 1042, 2042, 3042, 4042]
  gru s1: [42, 1042, 2042, 3042, 4042]
  gru s2: [42, 1042, 2042, 3042, 4042]
### Calibrated pooled OOF KL per seed
seed                      42      1042    2042    3042    4042
stage subset     model                                        
s1    all EEGs   mamba  0.7001  0.6786  0.7027  0.6902  0.6886
                 gru    0.7066  0.6983  0.6967  0.7037  0.6950
      >=10 votes mamba  0.5288  0.5184  0.5401  0.5272  0.5214
                 gru    0.5502  0.5401  0.5418  0.5480  0.5409
s2    >=10 votes mamba  0.4209  0.4066  0.4217  0.4077  0.4081
                 gru    0.4227  0.4166  0.4077  0.4173  0.4137

### Mean and SD across seeds
stage     subset model  n_seeds  kl_mean  kl_sd  kl_min  kl_max
   s1   all EEGs mamba        5   0.6921 0.0097  0.6786  0.7027
   s1   all EEGs   gru        5   0.7001 0.0049  0.6950  0.706

In [13]:
rows = []
PAIRS = [("mamba", "transformer"), ("mamba", "gru"), ("mamba", "cnn"), ("gru", "transformer"), ("gru", "cnn"), ("cnn", "transformer")]
for st, sname, ev, _ in CELLS:
    for a_, b_ in PAIRS:
        common = [s_ for s_ in SEEDS_ if (a_, st, s_, sname) in KL and (b_, st, s_, sname) in KL]
        if not common:
            continue
        tests = [(f"seed {s_}", KL[(a_, st, s_, sname)], KL[(b_, st, s_, sname)]) for s_ in common]
        if len(common) > 1:
            tests.append((f"mean of {len(common)} seeds", np.mean([KL[(a_, st, s_, sname)] for s_ in common], 0),
                          np.mean([KL[(b_, st, s_, sname)] for s_ in common], 0)))
        for lab, ka, kb in tests:
            m, lo, hi, p = cluster_bootstrap_diff((ka - kb)[ev], PID_OOF[ev])
            rows.append(dict(stage=st, subset=sname, comparison=f"{a_} - {b_}", seeds=lab, n=int(ev.sum()), kl_a=ka[ev].mean(), kl_b=kb[ev].mean(),
                             diff=m, ci_lo=lo, ci_hi=hi, rel_pct=100 * m / kb[ev].mean(), p=p))
comp = pd.DataFrame(rows)
if len(comp):
    main = comp["seeds"].str.startswith("mean")
    if not main.any():
        main = comp["seeds"] == "seed 42"
    comp["p_holm"] = np.nan
    comp.loc[main, "p_holm"] = holm(comp.loc[main, "p"].to_numpy())
    comp.to_csv(f"{CFG['RESULTS_DIR']}/v7_pairwise.csv", index=False)
    print("### Main tests: per-EEG KL averaged over seeds (negative = first model better)\n"
          + comp[main].drop(columns=["seeds"]).round(4).to_string(index=False))
    print("\n### Per-seed tests (consistency check: does the sign hold in every seed?)\n"
          + comp[~main].round(4).to_string(index=False))
print("outputs:", sorted(os.listdir(CFG["RESULTS_DIR"]))[-8:])

### Main tests: per-EEG KL averaged over seeds (negative = first model better)
stage     subset  comparison     n   kl_a   kl_b    diff   ci_lo   ci_hi  rel_pct      p  p_holm
   s1   all EEGs mamba - gru 17089 0.6921 0.7001 -0.0080 -0.0218  0.0072  -1.1456 0.2964  0.5928
   s1 >=10 votes mamba - gru  5600 0.5272 0.5442 -0.0170 -0.0300 -0.0037  -3.1255 0.0136  0.0408
   s2 >=10 votes mamba - gru  5600 0.4130 0.4156 -0.0026 -0.0110  0.0057  -0.6258 0.5340  0.5928

### Per-seed tests (consistency check: does the sign hold in every seed?)
stage     subset  comparison     seeds     n   kl_a   kl_b    diff   ci_lo   ci_hi  rel_pct      p  p_holm
   s1   all EEGs mamba - gru   seed 42 17089 0.7001 0.7066 -0.0065 -0.0275  0.0161  -0.9207 0.5404     NaN
   s1   all EEGs mamba - gru seed 1042 17089 0.6786 0.6983 -0.0197 -0.0350 -0.0045  -2.8258 0.0116     NaN
   s1   all EEGs mamba - gru seed 2042 17089 0.7027 0.6967  0.0060 -0.0095  0.0222   0.8665 0.4640     NaN
   s1   all EEGs mamba - gru s